### Importing The Required Libraries 

In [1]:
from typing import TypedDict, Annotated 
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langchain.tools import tool 
# from langchain_openai import ChatOpenAI 
from langgraph.graph import StateGraph,START,END
from langchain_google_genai import ChatGoogleGenerativeAI

C:\Users\rajve\miniconda3\envs\nlp_env\lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer
C:\Users\rajve\miniconda3\envs\nlp_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Defining AgentState

In [2]:
class AgentState(TypedDict):
    messages:Annotated[list,add_messages]

### Defining The Tool 

In [3]:
@tool
def inspect_order(order_id:str):
    """ This tool looks for shipping status of the order given order id of the customer """
    db={"ODR_1":"Shipped on 8 July", "ODR_2":"Ready to ship","ODR_3":"Will Deliver today"}
    return db.get(order_id,"No such  order")

In [4]:
tools=[inspect_order]

### Loading The LLM 

In [5]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

In [6]:
llm=llm.bind_tools(tools)

### Defining The Nodes 

In [7]:
def call_model(state:AgentState):
    print("LLM is Reasoning")
    response=llm.invoke(state["messages"])
    return {"messages": [response]}


### Defining The Router 

In [8]:
def should_continue(state:AgentState):
    ai_resp=state["messages"][-1]
    if ai_resp.tool_calls:
        print("AI requested a tool call,routing to the tool")
        return "tools"
    else:
        print("AI has finished answering routing to the end")
        return END

### Making The Graph Builder


In [9]:
builder=StateGraph(AgentState)
builder.add_node("Agent",call_model)
builder.add_node("tools",ToolNode(tools))

In [10]:
builder.add_edge(START,"Agent")
builder.add_conditional_edges("Agent",should_continue)
builder.add_edge("tools","Agent")

### Compiling the Graph

In [11]:
graph=builder.compile()

### Testing The Graph 

In [12]:
initial_state={"messages":"I want to know the status of the order ODR_1"}
final_state=graph.invoke(initial_state)

LLM is Reasoning
AI requested a tool call,routing to the tool
LLM is Reasoning
AI has finished answering routing to the end


In [13]:
final_state

{'messages': [HumanMessage(content='I want to know the status of the order ODR_1', additional_kwargs={}, response_metadata={}, id='c4010fa9-1845-4d56-9471-518614721613'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'inspect_order', 'arguments': '{"order_id": "ODR_1"}'}, '__gemini_function_call_thought_signatures__': {'26816af7-7039-4697-a5a6-ce89038321e7': 'Cq0CARFNMg+C1Jac8WW4CJwr7rvQw5f9RPEMLAcxqJSbjOToaPdOZirgOjX0xI72/R4RE+2DSQVYqgEWb8GESnIJ++TPge8+D+FfXZA0cXsnPKdw++KvzpD4Kdc012fzwv3W4Y6ttDHevSPizucy6mrbqSXPNvgkjv/3PrU8B31mKy795bCU7xxB35tCaCt3hnuDPU8i1FzmSwqXq/+1AxlAjAxaQWPJvwCwpQsuGlmI3R4g4ljAo8csybP28q36HYifMJyNY/zrBxBly8xqArusc3imrPYMw04Nb9EcVNChiW6vchjvCAWIOfZa4+Nn1rX4S0SG+V7PZ2mf09y7ylTluJk2aVyvTR0p+OXubymK2T2I9txyYID4fDbwWaSNmiygR5q7aAg4/kolzVMAhw=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f5abc-9a5b-7882-a904-902d23473bbe-0', tool_calls=[{'

In [14]:
initial_state={"messages":"I want to know the status of the order ODR_11"}
final_state=graph.invoke(initial_state)

LLM is Reasoning
AI requested a tool call,routing to the tool
LLM is Reasoning
AI has finished answering routing to the end


In [15]:
final_state

{'messages': [HumanMessage(content='I want to know the status of the order ODR_11', additional_kwargs={}, response_metadata={}, id='6e8b4a06-d753-4481-ae8f-a015d256063c'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'inspect_order', 'arguments': '{"order_id": "ODR_11"}'}, '__gemini_function_call_thought_signatures__': {'f53c96f2-b4f2-4783-98fd-055ed75f0295': 'CrYCARFNMg+NrCmXe+FgzRt3nBJ0nRKT7Jbm6fIMzEscjaHRUJOcEg/F3PrKMMrzMAf9k8ywABZR1SnH+CbhkNS+ogEbaWWc2tDRzQiADyJX+H2cskqRzZNYusKv2sC3B1D9jvIbCOBf2xfr1/2zbuByLF7EtkKIm1mdq+FpppiMsz8AJQZZZ2iI0+rM+tWrqSoGQIW1bo2SM89wSFo5dOSsFTI/0WsjZitH6sloFt6M983CUofkzuUSzZZ9G7htH6OV8rQ1hJHb11GLmMdIn+2xlgf2BCqAhU+KKOfvc8Enpab25zsRFTfmpJ4jnsSGcovP5LB2gVJ2L5vRadO6IHUnK6okh6D9tDBTmjRQdzNs4Bbz/ZVojrNKlbnd9QVX5Osssru26bFwicWea7eZNpzoif4f5ZMA4A=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f5abd-3ce7-7af0-9a82-cdef5ef7689e-0', 

In [ ]:
s